# Phase 8 — Analyse de risque

Ce notebook lit les fichiers déjà publiés. Il ne réestime pas les poids et il ne recalcule pas la VaR. Les conventions d'unité, de signe et d'échantillon, les formules, les limites et la reconstruction de la page sont dans `documentation/08_risk_analytics.ipynb`. Le rapport chiffré est `reports/risk/08_rapport_risque.ipynb`.

`reports/risk/dashboard.html` est réécrite par `build_risk`, donc par le rafraîchissement du pipeline. Ce notebook ne la réécrit pas.

Trois questions :

1. La volatilité, la VaR et l'expected shortfall classent-ils les trois livres de la même façon ?
2. La corrélation moyenne décrit-elle les paires, ou un secteur se détache-t-il ?
3. Ce secteur, et les paires au-dessus du seuil de lecture, pèsent-ils dans les poids du mois de rapport ?

Les seuils de lecture sont `PAIR_LIST_FLOOR` et `PAIR_LIST_TAIL` dans `src/risk/build_risk.py`. Ce ne sont pas des plafonds de l'IPS. Les noms, les mois et les pourcentages imprimés viennent des fichiers de cette exécution.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import plotly.graph_objects as go

from src.portfolio_construction.build_portfolios import MEMBERSHIP_FILE, WEIGHTS_FILE
from src.portfolio_monitoring.build_monitoring import (
    CONCENTRATION_FILE,
    REPORT_FILE,
    SECTOR_FILE,
    load_holding_classification,
)
from src.risk.build_risk import (
    MATRIX_FILE,
    METRICS_OUTPUT,
    PAIR_LIST_FLOOR,
    PAIR_LIST_TAIL,
    SUMMARY_FILE,
    TAIL_FILE,
)
from src.risk.dashboard import apply_layout, tail_bars, tail_months

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}
READING_FLOOR = PAIR_LIST_FLOOR
READING_TAIL = PAIR_LIST_TAIL


def percent(value):
    return f"{float(value) * 100:.1f}".replace(".", ",") + " %"


metrics = pd.read_csv(METRICS_OUTPUT)
metrics = metrics.loc[metrics["strategy"].isin(LABELS)].copy()
tail = pd.read_csv(TAIL_FILE)
tail = tail.loc[tail["strategy"].isin(LABELS)].copy()
summary = pd.read_csv(SUMMARY_FILE).iloc[0]
matrix = pd.read_csv(MATRIX_FILE, index_col="symbol")
membership = pd.read_csv(MEMBERSHIP_FILE)
weights = pd.read_csv(WEIGHTS_FILE)
concentration = pd.read_csv(CONCENTRATION_FILE)
sectors = pd.read_csv(SECTOR_FILE)
report = pd.read_csv(REPORT_FILE)

included = membership.loc[
    membership["decision"].eq("included"),
    ["yahoo_symbol", "ticker", "name"],
].copy()
classified = load_holding_classification()[["ticker", "sector"]]
identity = included.merge(classified, on="ticker", how="left")
if identity["sector"].isna().any() or identity["yahoo_symbol"].duplicated().any():
    raise RuntimeError("A included name has no sector, or the ticker is ambiguous.")
if set(identity["yahoo_symbol"]) != set(matrix.columns):
    raise RuntimeError("The correlation matrix does not match the included names.")
window_end = str(metrics["window_end"].iloc[0])
if set(report["report_month"].astype(str)) != {window_end}:
    raise RuntimeError("The monitoring report month is not the risk window end.")
if str(weights["month"].max()) != window_end:
    raise RuntimeError("The weight file does not reach the risk window end.")

symbols = list(matrix.columns)
pairs = []
for i, left in enumerate(symbols):
    for right in symbols[i + 1 :]:
        pairs.append((left, right, float(matrix.loc[left, right])))
pairs = pd.DataFrame(pairs, columns=["left", "right", "correlation"])
lookup = identity.set_index("yahoo_symbol")
pairs["left_name"] = pairs["left"].map(lookup["name"])
pairs["right_name"] = pairs["right"].map(lookup["name"])
pairs["left_sector"] = pairs["left"].map(lookup["sector"])
pairs["right_sector"] = pairs["right"].map(lookup["sector"])
pairs["same_sector"] = pairs["left_sector"].eq(pairs["right_sector"])

print(f"Période {metrics['window_start'].iloc[0]} – {window_end}, {int(summary['months'])} mois.")
print(
    f"{int(summary['names'])} titres inclus, {len(pairs)} paires. "
    f"Corrélation moyenne {percent(summary['average_correlation'])}. "
    f"Médiane {percent(pairs['correlation'].median())}."
)
print(
    f"Paires négatives : {(pairs['correlation'] < 0).sum()}. "
    f"Paires d'au moins {percent(READING_FLOOR)} : {(pairs['correlation'] >= READING_FLOOR).sum()}. "
    f"Paires d'au moins {percent(READING_TAIL)} : {(pairs['correlation'] >= READING_TAIL).sum()}."
)


Période 2016-11 – 2026-10, 120 mois.
28 titres inclus, 378 paires. Corrélation moyenne 16,3 %. Médiane 14,5 %.
Paires négatives : 43. Paires d'au moins 50,0 % : 13. Paires d'au moins 70,0 % : 4.


## La volatilité et la queue

La volatilité mesure l'écart de tous les mois autour de leur moyenne. La VaR retient un seul rang parmi les mois les plus mauvais. L'expected shortfall moyenne ces mois. Un livre peut être le plus calme sur l'ensemble de la période et ne pas avoir la queue la plus douce.

La cellule nomme, à partir de `risk_by_strategy.csv`, le livre à la volatilité la plus basse, celui à la VaR la plus profonde, et celui à l'expected shortfall le plus doux. Elle imprime aussi, pour chaque livre, de combien l'expected shortfall est plus profond que la VaR.

Elle compte ensuite les mois de `var_tail.csv` selon le nombre de livres qui les partagent. Un mois présent dans tous les livres est une date de la liste. Un mois présent dans un seul livre tient à la règle de poids de ce livre.


In [2]:
print("Volatilité, VaR 95 % et expected shortfall 95 %.")
for strategy, label in LABELS.items():
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    print(
        f"{label} : volatilité {percent(row['annualized_volatility'])}, "
        f"VaR {percent(row['var_95'])}, "
        f"expected shortfall {percent(row['expected_shortfall_95'])}."
    )
lowest = metrics.loc[metrics["annualized_volatility"].astype(float).idxmin()]
deepest = metrics.loc[metrics["var_95"].astype(float).idxmin()]
mildest = metrics.loc[metrics["expected_shortfall_95"].astype(float).idxmax()]
print(
    f"Volatilité la plus basse : {LABELS[lowest['strategy']]} "
    f"({percent(lowest['annualized_volatility'])})."
)
print(
    f"VaR la plus profonde : {LABELS[deepest['strategy']]} ({percent(deepest['var_95'])})."
)
print(
    f"Expected shortfall le plus doux : {LABELS[mildest['strategy']]} "
    f"({percent(mildest['expected_shortfall_95'])})."
)
if lowest["strategy"] != deepest["strategy"]:
    print("Le livre à la volatilité la plus basse n'a pas la VaR la plus profonde.")
if lowest["strategy"] == mildest["strategy"]:
    print("Le livre à la volatilité la plus basse a aussi l'expected shortfall le plus doux.")

print("Profondeur de la queue, expected shortfall contre VaR :")
for strategy, label in LABELS.items():
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    gap = float(row["var_95"]) - float(row["expected_shortfall_95"])
    print(f"  {label} : l'expected shortfall est plus profond de {percent(gap)}.")

presence = {}
for strategy in LABELS:
    for month in tail.loc[tail["strategy"].eq(strategy), "month"].astype(str):
        presence.setdefault(month, set()).add(strategy)
print("Mois de queue selon le nombre de livres :")


def book_phrase(count, total):
    if count == total:
        return "Dans tous les livres"
    if count == 1:
        return "Dans un seul livre"
    return f"Dans {count} livres"


for count in range(len(LABELS), 0, -1):
    months = sorted(month for month, books in presence.items() if len(books) == count)
    if months:
        print(f"  {book_phrase(count, len(LABELS))} : {', '.join(months)}.")

labeled = tail.copy()
labeled["strategy"] = labeled["strategy"].map(LABELS)
metrics_labeled = metrics.copy()
metrics_labeled["strategy"] = metrics_labeled["strategy"].map(LABELS)
tail_bars(metrics_labeled).show()
tail_months(labeled).show()


Volatilité, VaR 95 % et expected shortfall 95 %.
Équipondéré : volatilité 12,3 %, VaR -5,5 %, expected shortfall -7,9 %.
Volatilité minimale : volatilité 11,8 %, VaR -5,6 %, expected shortfall -7,6 %.
Sharpe maximal : volatilité 12,1 %, VaR -5,9 %, expected shortfall -8,2 %.
Volatilité la plus basse : Volatilité minimale (11,8 %).
VaR la plus profonde : Sharpe maximal (-5,9 %).
Expected shortfall le plus doux : Volatilité minimale (-7,6 %).
Le livre à la volatilité la plus basse n'a pas la VaR la plus profonde.
Le livre à la volatilité la plus basse a aussi l'expected shortfall le plus doux.
Profondeur de la queue, expected shortfall contre VaR :
  Équipondéré : l'expected shortfall est plus profond de 2,4 %.
  Volatilité minimale : l'expected shortfall est plus profond de 2,1 %.
  Sharpe maximal : l'expected shortfall est plus profond de 2,3 %.
Mois de queue selon le nombre de livres :
  Dans tous les livres : 2020-02, 2020-03, 2020-10, 2022-01, 2022-09.
  Dans 2 livres : 2018-10.
  D

## Ce que la moyenne ne montre pas

La corrélation de Pearson d'une paire résume les mois de la fenêtre, chacun avec le même poids. La moyenne hors diagonale résume toutes les paires, chacune une fois. Une moyenne modérée peut côtoyer un secteur dont les lignes bougent ensemble.

La cellule sépare les paires d'un même secteur et les paires de deux secteurs. Le secteur le plus lié est celui dont la moyenne interne est la plus haute, parmi les secteurs qui ont au moins deux paires. Une seule paire ne suffit pas à qualifier un bloc.

Le secteur vient du même classement que la phase 6, joint par ticker. Le module de risque n'écrit pas ce secteur dans la matrice : la matrice publiée est un panneau de titres.

La cellule imprime ensuite les poids de secteur au mois de rapport, puis le poids du secteur le plus lié. Un bloc très corrélé et lourd dans le livre n'est pas le même fait qu'un bloc très corrélé et peu tenu.


In [3]:
same = pairs.loc[pairs["same_sector"]]
cross = pairs.loc[~pairs["same_sector"]]
print(
    f"Paires du même secteur : {len(same)}, corrélation moyenne {percent(same['correlation'].mean())}."
)
print(
    f"Paires de deux secteurs différents : {len(cross)}, "
    f"corrélation moyenne {percent(cross['correlation'].mean())}."
)
print("Corrélation interne par secteur :")
sector_mean = {}
for sector, block in same.groupby("left_sector"):
    pair_word = "paire" if len(block) == 1 else "paires"
    sector_mean[sector] = float(block["correlation"].mean())
    print(
        f"  {sector} : {len(block)} {pair_word}, "
        f"moyenne {percent(sector_mean[sector])}, "
        f"maximum {percent(block['correlation'].max())}."
    )
linked = same.groupby("left_sector").size()
linked = linked.loc[linked.ge(2)]
if linked.empty:
    raise RuntimeError("No sector has two pairs.")
tight_sector = max(linked.index, key=lambda sector: sector_mean[sector])
print(
    f"Secteur le plus lié, parmi ceux qui ont au moins deux paires : {tight_sector}, "
    f"moyenne {percent(sector_mean[tight_sector])}."
)
strong = pairs.loc[pairs["correlation"].ge(READING_FLOOR)]
print(f"Paires d'au moins {percent(READING_FLOOR)}, par secteur :")
for sector, block in strong.loc[strong["same_sector"]].groupby("left_sector"):
    print(f"  {sector} : {len(block)}.")
cross_strong = int((~strong["same_sector"]).sum())
if cross_strong:
    print(f"  deux secteurs différents : {cross_strong}.")
print("Poids par secteur au mois de rapport :")
for strategy, label in LABELS.items():
    block = sectors.loc[sectors["strategy"].eq(strategy)].sort_values(
        "weight", ascending=False, kind="mergesort"
    )
    listed = ", ".join(
        f"{row.sector} {percent(row.weight)}" for row in block.itertuples(index=False)
    )
    held = concentration.loc[concentration["strategy"].eq(strategy)].iloc[0]
    effective = str(round(float(held["effective_holdings"]), 1)).replace(".", ",")
    print(f"  {label} : {listed}.")
    print(
        f"  {int(held['invested_names'])} lignes investies, nombre effectif {effective}."
    )

print(f"Poids de {tight_sector} au mois de rapport :")
for strategy, label in LABELS.items():
    block = sectors.loc[
        sectors["strategy"].eq(strategy) & sectors["sector"].eq(tight_sector),
        "weight",
    ]
    if len(block) != 1:
        raise RuntimeError(f"{strategy} has no unique weight for {tight_sector}.")
    print(f"  {label} : {percent(float(block.iloc[0]))}.")


Paires du même secteur : 94, corrélation moyenne 24,6 %.
Paires de deux secteurs différents : 284, corrélation moyenne 13,6 %.
Corrélation interne par secteur :
  Consumer Staples : 1 paire, moyenne 12,4 %, maximum 12,4 %.
  Health Care : 45 paires, moyenne 14,5 %, maximum 62,9 %.
  Industrials : 3 paires, moyenne -1,3 %, maximum 9,4 %.
  Real Estate : 45 paires, moyenne 36,7 %, maximum 76,8 %.
Secteur le plus lié, parmi ceux qui ont au moins deux paires : Real Estate, moyenne 36,7 %.
Paires d'au moins 50,0 %, par secteur :
  Health Care : 1.
  Real Estate : 12.
Poids par secteur au mois de rapport :
  Équipondéré : Health Care 35,7 %, Real Estate 35,7 %, Industrials 10,7 %, Consumer Staples 7,1 %, Consumer Discretionary 3,6 %, Information Technology 3,6 %, Utilities 3,6 %.
  28 lignes investies, nombre effectif 28,0.
  Volatilité minimale : Health Care 40,0 %, Real Estate 31,6 %, Industrials 11,1 %, Consumer Discretionary 5,0 %, Consumer Staples 4,9 %, Information Technology 4,3 %, Ut

## Lire la matrice

Les axes portent le ticker, pour que les cases restent larges. Le survol donne le nom et le secteur. L'ordre est le secteur, puis le nom : un groupe qui bouge ensemble apparaît comme un carré.

Le rouge est une corrélation négative, le bleu une corrélation élevée. L'échelle est fixe, de −1 à +1, la même que la feuille Corrélation d'Excel et que la matrice Power BI. La phrase sous le graphique reprend le secteur le plus lié déjà calculé par la cellule précédente. Elle ne le choisit pas d'après la couleur.


In [4]:
order = identity.sort_values(["sector", "name"], kind="mergesort")
ordered_symbols = order["yahoo_symbol"].tolist()
ordered = matrix.loc[ordered_symbols, ordered_symbols]
axis = order["ticker"].astype(str).tolist()
hover = [f"{row.ticker} — {row.name} | {row.sector}" for row in order.itertuples(index=False)]
labels = [
    [f"{row_label}<br>{column_label}" for column_label in hover]
    for row_label in hover
]
figure = go.Figure(
    data=go.Heatmap(
        z=ordered.to_numpy(),
        x=axis,
        y=axis,
        customdata=labels,
        zmin=-1,
        zmax=1,
        xgap=2,
        ygap=2,
        colorscale="RdBu",
        colorbar={"title": "Corrélation", "tickformat": ".0%", "len": 0.8},
        hovertemplate="%{customdata}<br>%{z:.1%}<extra></extra>",
    )
)
figure.update_xaxes(tickangle=90, title="", tickfont={"size": 14}, automargin=True)
figure.update_yaxes(
    autorange="reversed",
    title="",
    tickfont={"size": 14},
    automargin=True,
    scaleanchor="x",
)
apply_layout(figure, "Matrice de corrélation des titres inclus", 1400)
figure.update_layout(
    width=1500,
    autosize=False,
    margin={"l": 120, "b": 140, "r": 40, "t": 80},
)
figure.show()
print(f"L'ordre des axes suit le secteur, puis le nom. Le secteur le plus lié est {tight_sector}.")


L'ordre des axes suit le secteur, puis le nom. Le secteur le plus lié est Real Estate.


## Ces paires dans les poids du mois de rapport

Le premier seuil de lecture retient les paires à commenter. Pour chacune, la cellule lit le poids publié de chaque ligne au mois de rapport, et la somme des deux.

Une paire très liée et presque absente du livre est une dépendance de la liste. Une paire très liée et tenue est une dépendance à l'intérieur du livre, surtout si une ligne est proche du plafond par émetteur de l'IPS. La cellule imprime les poids. Elle ne décide pas à l'avance quelle paire sera la plus lourde : elle la désigne en comparant les sommes.


In [5]:
report_weights = weights.loc[weights["month"].astype(str).eq(window_end)]


def held_weight(strategy, symbol):
    rows = report_weights.loc[
        report_weights["strategy"].eq(strategy) & report_weights["yahoo_symbol"].eq(symbol),
        "weight",
    ]
    if len(rows) != 1:
        raise RuntimeError(f"{strategy} has no unique weight for {symbol} in {window_end}.")
    return float(rows.iloc[0])


print(f"Paires d'au moins {percent(READING_FLOOR)}, et leur poids en {window_end}.")
selected = pairs.loc[pairs["correlation"].ge(READING_FLOOR)].sort_values(
    "correlation", ascending=False, kind="mergesort"
)
for row in selected.itertuples(index=False):
    print(
        f"{percent(row.correlation)}  {row.left_name} et {row.right_name}, {row.left_sector}."
    )
    parts = []
    for strategy, label in LABELS.items():
        left_weight = held_weight(strategy, row.left)
        right_weight = held_weight(strategy, row.right)
        parts.append(
            f"{label} {percent(left_weight)} et {percent(right_weight)} "
            f"(somme {percent(left_weight + right_weight)})"
        )
    print("  " + " ; ".join(parts) + ".")

print(f"Paire la plus lourde d'au moins {percent(READING_FLOOR)}, par livre :")
for strategy, label in LABELS.items():
    totals = []
    for row in selected.itertuples(index=False):
        total = held_weight(strategy, row.left) + held_weight(strategy, row.right)
        totals.append((total, row.correlation, row.left_name, row.right_name))
    total, correlation, left_name, right_name = max(totals)
    print(
        f"  {label} : {left_name} et {right_name}, "
        f"{percent(correlation)}, somme {percent(total)}."
    )


Paires d'au moins 50,0 %, et leur poids en 2026-10.
76,8 %  COVIVIO SA et GECINA SA, Real Estate.
  Équipondéré 3,6 % et 3,6 % (somme 7,1 %) ; Volatilité minimale 0,0 % et 2,7 % (somme 2,7 %) ; Sharpe maximal 3,6 % et 1,2 % (somme 4,8 %).
71,7 %  GECINA SA et LAND SECURITIES GROUP REIT PLC, Real Estate.
  Équipondéré 3,6 % et 3,6 % (somme 7,1 %) ; Volatilité minimale 2,7 % et 1,8 % (somme 4,5 %) ; Sharpe maximal 1,2 % et 4,9 % (somme 6,1 %).
71,6 %  COVIVIO SA et LAND SECURITIES GROUP REIT PLC, Real Estate.
  Équipondéré 3,6 % et 3,6 % (somme 7,1 %) ; Volatilité minimale 0,0 % et 1,8 % (somme 1,8 %) ; Sharpe maximal 3,6 % et 4,9 % (somme 8,5 %).
71,4 %  DIGITAL REALTY TRUST REIT et EQUINIX REIT, Real Estate.
  Équipondéré 3,6 % et 3,6 % (somme 7,1 %) ; Volatilité minimale 5,0 % et 1,6 % (somme 6,6 %) ; Sharpe maximal 4,6 % et 4,6 % (somme 9,3 %).
64,9 %  COVIVIO SA et NEPI ROCKCASTLE, Real Estate.
  Équipondéré 3,6 % et 3,6 % (somme 7,1 %) ; Volatilité minimale 0,0 % et 4,0 % (somme 4,

## Lecture

On lit d'abord si le livre le moins variable est aussi celui dont la queue est la plus douce, et de combien l'expected shortfall s'écarte de la VaR.

On lit ensuite si le secteur le plus lié est aussi un secteur lourd au mois de rapport. La matrice rend ce bloc visible : les cases d'un même secteur sont voisines, et le survol nomme les titres.

On lit enfin la liste des paires au-dessus du seuil. La paire la plus lourde de chaque livre est celle dont la somme des deux poids est la plus haute dans cette liste.

Ces phrases sont produites par les cellules. Une reconstruction avec d'autres cours les réécrit. Le texte markdown ne contient ni nom de titre, ni mois de la queue, ni pourcentage mesuré.
